In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("panel_weekly.csv")

print(df.shape)
print(df.head())
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'panel_weekly.csv'

In [2]:
# 1. Vérifier les valeurs manquantes
print("Valeurs manquantes :")
print(df.isnull().sum())

# 2. Vérifier les doublons
print("\nDoublons :", df.duplicated().sum())

# 3. Vérifier qu'il n'existe qu'une observation
# par matière première et par semaine
print(
    "\nDoublons week + commodity :",
    df.duplicated(subset=["week", "commodity"]).sum()
)

# 4. Statistiques descriptives
print("\nStatistiques descriptives :")
display(df.describe())

# 5. Vérifier les périodes couvertes
print("\nPremière semaine :", df["week"].min())
print("Dernière semaine :", df["week"].max())

Valeurs manquantes :
week          0
commodity     0
ret           0
mom_4w        0
mom_12w       0
vol_12w       0
target        0
net_tone      0
n_relevant    0
has_news      0
attention     0
dtype: int64

Doublons : 0

Doublons week + commodity : 0

Statistiques descriptives :


,ret,mom_4w,mom_12w,vol_12w,target,net_tone,n_relevant,has_news,attention
count,4365.000000,4365.000000,4365.000000,4365.000000,4365.000000,4365.000000,4365.000000,4365.000000,4365.000000
mean,0.001686,0.006809,0.020141,0.042286,0.001674,0.018959,2.862772,0.536999,0.752098
std,0.050091,0.096042,0.158501,0.027660,0.050126,0.458186,11.220693,0.498686,0.884284
min,-0.293098,-0.763677,-1.022311,0.007624,-0.293098,-1.000000,0.000000,0.000000,0.000000
25%,-0.022130,-0.040647,-0.058624,0.024308,-0.022136,0.000000,0.000000,0.000000,0.000000
50%,0.001166,0.004922,0.016250,0.035090,0.001160,0.000000,1.000000,1.000000,0.693147
75%,0.023610,0.053251,0.094552,0.051270,0.023610,0.166667,3.000000,1.000000,1.386294
max,0.699968,0.737780,0.955557,0.249565,0.699968,1.000000,401.000000,1.000000,5.996452



Première semaine : 2010-01-01
Dernière semaine : 2026-09-18


In [3]:
# Conversion des dates
df["week"] = pd.to_datetime(df["week"])

# Rendements extrêmes (> 30 % ou < -30 %)
extreme_returns = df[
    (df["ret"].abs() > 0.30) |
    (df["target"].abs() > 0.30)
]

print("Nombre de rendements extrêmes :",
      len(extreme_returns))

display(
    extreme_returns[
        ["week", "commodity", "ret", "target"]
    ].sort_values("week")
)

# Semaines ayant plus de 100 articles pertinents
extreme_news = df[df["n_relevant"] > 100]

print("\nObservations avec plus de 100 articles :",
      len(extreme_news))

display(
    extreme_news[
        ["week", "commodity", "n_relevant", "net_tone"]
    ].sort_values("n_relevant", ascending=False)
)

Nombre de rendements extrêmes : 10


,week,commodity,ret,target
2670,2020-03-27,crude_oil,-0.041016,0.317527
2675,2020-04-03,crude_oil,0.317527,-0.196895
3174,2022-02-25,wheat,0.057716,0.599051
3179,2022-03-04,wheat,0.599051,-0.191395
3738,2024-04-26,natural_gas,-0.078767,0.327138
3743,2024-05-03,natural_gas,0.327138,0.051354
4188,2026-01-16,natural_gas,-0.020827,0.699968
4193,2026-01-23,natural_gas,0.699968,-0.174597
4215,2026-02-27,crude_oil,0.009489,0.356312
4220,2026-03-06,crude_oil,0.356312,0.085919



Observations avec plus de 100 articles : 12


,week,commodity,n_relevant,net_tone
4225,2026-03-13,crude_oil,401.0,0.590524
4220,2026-03-06,crude_oil,220.0,0.793636
4265,2026-05-08,crude_oil,191.0,-0.249215
4295,2026-06-19,crude_oil,184.0,-0.616848
4230,2026-03-20,crude_oil,160.0,0.565625
4320,2026-07-24,crude_oil,159.0,0.594969
4310,2026-07-10,crude_oil,132.0,0.328030
4315,2026-07-17,crude_oil,127.0,0.638583
4270,2026-05-15,crude_oil,121.0,0.468595
4260,2026-05-01,crude_oil,118.0,0.486441


In [4]:
# Vérification de la couverture des actualités

df["week"] = pd.to_datetime(df["week"])
df["year"] = df["week"].dt.year

coverage = df.groupby(
    ["year", "commodity"]
).agg(
    weeks=("week", "nunique"),
    weeks_with_news=("has_news", "sum"),
    total_articles=("n_relevant", "sum"),
    max_articles=("n_relevant", "max")
).reset_index()

# Pourcentage de semaines avec actualités
coverage["coverage_pct"] = (
    100 * coverage["weeks_with_news"]
    / coverage["weeks"]
).round(2)

# Tableau de comparaison entre années
coverage_table = coverage.pivot(
    index="year",
    columns="commodity",
    values="coverage_pct"
)

display(coverage_table)

commodity,copper,crude_oil,gold,natural_gas,wheat
year,,,,,
2010,32.08,32.08,86.79,3.77,1.89
2011,48.08,46.15,86.54,15.38,7.69
2012,42.31,32.69,80.77,5.77,0.00
2013,38.46,34.62,88.46,9.62,0.00
2014,26.92,38.46,86.54,3.85,0.00
2015,36.54,65.38,82.69,13.46,1.92
2016,39.62,60.38,98.11,9.43,1.89
2017,78.85,67.31,100.00,7.69,11.54
2018,59.62,96.15,98.08,94.23,50.00


In [5]:
import pandas as pd

# 1. Convertir les dates
df["week"] = pd.to_datetime(df["week"])

# 2. Trier les données chronologiquement
df = df.sort_values(["week", "commodity"]).copy()

# 3. Date correspondant au rendement à prédire
df["target_week"] = df["week"] + pd.Timedelta(days=7)

# 4. Séparation chronologique
train = df[df["target_week"] < "2021-01-01"].copy()

val = df[
    (df["week"] >= "2021-01-01") &
    (df["target_week"] < "2024-01-01")
].copy()

test = df[df["week"] >= "2024-01-01"].copy()

# 5. Afficher les résultats
print("Train :", train.shape)
print("Validation :", val.shape)
print("Test :", test.shape)

print("\nPériode Train :", train["week"].min(), train["week"].max())
print("Période Validation :", val["week"].min(), val["week"].max())
print("Période Test :", test["week"].min(), test["week"].max())

Train : (2865, 13)
Validation : (780, 13)
Test : (710, 13)

Période Train : 2010-01-01 00:00:00 2020-12-18 00:00:00
Période Validation : 2021-01-01 00:00:00 2023-12-22 00:00:00
Période Test : 2024-01-05 00:00:00 2026-09-18 00:00:00


In [6]:
# Variables explicatives
features = [
    "commodity",
    "ret",
    "mom_4w",
    "mom_12w",
    "vol_12w"
]

# Variable à prédire
target = "target"

# Train
X_train = train[features].copy()
y_train = train[target].copy()

# Validation
X_val = val[features].copy()
y_val = val[target].copy()

# Test
X_test = test[features].copy()
y_test = test[target].copy()

# Vérification
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

print("\nPremières lignes de X_train :")
display(X_train.head())

print("\nPremières valeurs de y_train :")
display(y_train.head())

X_train : (2865, 5)
y_train : (2865,)

Premières lignes de X_train :


,commodity,ret,mom_4w,mom_12w,vol_12w
2,copper,0.017273,0.036664,0.168475,0.030087
0,crude_oil,0.016784,0.056464,0.112384,0.045818
1,gold,-0.008061,-0.064041,0.047588,0.024435
3,natural_gas,-0.012582,0.209087,0.197335,0.087556
4,wheat,0.032412,0.010031,0.163318,0.054906



Premières valeurs de y_train :


2    0.018182
0    0.042717
1    0.039262
3    0.031766
4    0.049861
Name: target, dtype: float64

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Variables numériques
numeric_features = [
    "ret",
    "mom_4w",
    "mom_12w",
    "vol_12w"
]

# Variable catégorielle
categorical_features = ["commodity"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# Pipeline complet : Preprocessing + Ridge
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

print("Pipeline Ridge créé !")

Pipeline Ridge créé !


In [8]:
# Entraînement du modèle
ridge_pipeline.fit(X_train, y_train)

# Prédictions sur la validation
y_pred_val = ridge_pipeline.predict(X_val)

print("Modèle Ridge entraîné !")

# Comparaison des premières prédictions
comparison = pd.DataFrame({
    "Real Return": y_val.values,
    "Predicted Return": y_pred_val
})

display(comparison.head(10))

Modèle Ridge entraîné !


,Real Return,Predicted Return
0,0.045390,-0.003089
1,0.076669,-0.001655
2,-0.031502,0.002158
3,0.063411,0.005884
4,-0.002732,-0.001233
5,-0.019192,-0.000842
6,0.002297,0.000798
7,-0.002997,0.001009
8,0.013704,0.007787
9,0.057534,-0.001015


In [9]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Prédictions Ridge
y_pred_val = ridge_pipeline.predict(X_val)

# Fonction pour calculer les métriques
def evaluate_model(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    directional_accuracy = np.mean(
        (np.array(y_true) > 0) ==
        (np.array(y_pred) > 0)
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Directional Accuracy": directional_accuracy
    }

# Évaluer Ridge
ridge_results = evaluate_model(y_val, y_pred_val)

# Baseline : prédire toujours la moyenne du Train
baseline_pred = np.full(
    len(y_val),
    y_train.mean()
)

baseline_results = evaluate_model(
    y_val, baseline_pred
)

# Tableau comparatif
results = pd.DataFrame(
    [baseline_results, ridge_results],
    index=["Baseline (Train Mean)", "Ridge"]
)

display(results.round(4))

,MAE,RMSE,R2,Directional Accuracy
Baseline (Train Mean),0.0382,0.0582,-0.0007,0.5256
Ridge,0.0385,0.0588,-0.0234,0.5244


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Création du modèle Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Entraînement
rf_pipeline.fit(X_train, y_train)

# Prédictions sur la validation
y_pred_rf = rf_pipeline.predict(X_val)

# Évaluation
rf_results = evaluate_model(y_val, y_pred_rf)

# Comparaison
results = pd.DataFrame(
    [baseline_results, ridge_results, rf_results],
    index=["Baseline", "Ridge", "Random Forest"]
)

display(results.round(4))

,MAE,RMSE,R2,Directional Accuracy
Baseline,0.0382,0.0582,-0.0007,0.5256
Ridge,0.0385,0.0588,-0.0234,0.5244
Random Forest,0.0387,0.0590,-0.0291,0.4974


In [12]:
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# Création du modèle LightGBM
lgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=20,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

# Entraînement sur le Train
lgbm_pipeline.fit(X_train, y_train)

# Prédictions sur la Validation
y_pred_lgbm = lgbm_pipeline.predict(X_val)

# Calcul des métriques
lgbm_results = evaluate_model(y_val, y_pred_lgbm)

print("LightGBM entraîné !")

LightGBM entraîné !


In [13]:
# Comparaison finale de l'expérience 1
results = pd.DataFrame(
    [
        baseline_results,
        ridge_results,
        rf_results,
        lgbm_results
    ],
    index=[
        "Baseline",
        "Ridge",
        "Random Forest",
        "LightGBM"
    ]
)

display(results.round(4))

,MAE,RMSE,R2,Directional Accuracy
Baseline,0.0382,0.0582,-0.0007,0.5256
Ridge,0.0385,0.0588,-0.0234,0.5244
Random Forest,0.0387,0.0590,-0.0291,0.4974
LightGBM,0.0391,0.0600,-0.0650,0.4936


In [14]:
# Comparaison Train vs Validation

models = {
    "Ridge": ridge_pipeline,
    "Random Forest": rf_pipeline,
    "LightGBM": lgbm_pipeline
}

overfitting_results = []

for name, model in models.items():

    # Prédictions sur le Train
    y_pred_train = model.predict(X_train)

    # Prédictions sur la Validation
    y_pred_val = model.predict(X_val)

    # RMSE
    train_rmse = np.sqrt(
        mean_squared_error(y_train, y_pred_train)
    )

    val_rmse = np.sqrt(
        mean_squared_error(y_val, y_pred_val)
    )

    overfitting_results.append({
        "Model": name,
        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,
        "Gap": val_rmse - train_rmse
    })

overfitting_df = pd.DataFrame(overfitting_results)

display(overfitting_df.round(4))

,Model,Train RMSE,Validation RMSE,Gap
0,Ridge,0.0435,0.0588,0.0153
1,Random Forest,0.0428,0.0590,0.0162
2,LightGBM,0.0416,0.0600,0.0185
